# 02) Fine-tune `microsoft/codebert-base` (LoRA) for vulnerability detection

Input expected from notebook 01: `codebert_dataset.csv`

This notebook trains a binary classifier with PEFT LoRA and saves adapters + tokenizer.

> **Kaggle Setup:** This notebook is configured for Kaggle. Enable **GPU T4 ×2** or **P100** accelerator from *Settings → Accelerator*.
>
> **Data Input:** Add the output of **notebook 01** (or a Kaggle dataset containing `codebert_dataset.csv`) as an input dataset under
> *Add Data → Your Datasets / Notebook Output*. Update `DATASET_SLUG` below to match the name you gave it.

In [ ]:
!pip -q install transformers datasets peft accelerate scikit-learn evaluate

In [ ]:
# ── Environment & GPU check ──
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU device      : {torch.cuda.get_device_name(0)}')
    print(f'GPU memory      : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
    torch.cuda.empty_cache()

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device    : {DEVICE}')

assert torch.cuda.is_available(), (
    'GPU not detected! Go to Settings → Accelerator and select GPU T4 ×2 or P100.'
)

In [ ]:
import os, glob
import numpy as np
import pandas as pd
from pathlib import Path
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, accuracy_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from peft import LoraConfig, TaskType, get_peft_model

MODEL_NAME = 'microsoft/codebert-base'
MAX_LEN = 256

# ── Kaggle paths ──
# Change DATASET_SLUG to match the name of your uploaded dataset / notebook-output.
# For example, if you added notebook-01's output as input, it will appear under
# /kaggle/input/<your-notebook-slug>/codebert_dataset.csv
DATASET_SLUG = 'codebert-dataset-cvefixes'  # ← update this to match your input name

INPUT_DIR  = Path('/kaggle/input') / DATASET_SLUG
OUT_DIR    = Path('/kaggle/working/codebert_lora')

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Auto-detect the CSV: look inside input dir (handles nested folders too)
csv_candidates = list(INPUT_DIR.rglob('codebert_dataset.csv'))
if csv_candidates:
    CSV_PATH = csv_candidates[0]
else:
    # Fallback: scan all input dirs (useful when slug name is unknown)
    fallback = list(Path('/kaggle/input').rglob('codebert_dataset.csv'))
    assert fallback, (
        'codebert_dataset.csv not found in /kaggle/input/. '
        'Please add the output of notebook 01 (or a dataset containing codebert_dataset.csv) as input data.'
    )
    CSV_PATH = fallback[0]

print(f'CSV_PATH : {CSV_PATH}')
print(f'OUT_DIR  : {OUT_DIR}')

In [ ]:
df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=['code', 'label'])
df['label'] = df['label'].astype(int)

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

train_ds = Dataset.from_pandas(train_df[['code', 'label']], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[['code', 'label']], preserve_index=False)
test_ds = Dataset.from_pandas(test_df[['code', 'label']], preserve_index=False)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch['code'], truncation=True, max_length=MAX_LEN)

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

train_ds = train_ds.rename_column('label', 'labels')
val_ds = val_ds.rename_column('label', 'labels')
test_ds = test_ds.rename_column('label', 'labels')

cols = ['input_ids', 'attention_mask', 'labels']
train_ds.set_format(type='torch', columns=cols)
val_ds.set_format(type='torch', columns=cols)
test_ds.set_format(type='torch', columns=cols)

In [ ]:
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=['query', 'value']
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    acc = accuracy_score(labels, preds)
    try:
        auc = roc_auc_score(labels, probs[:, 1])
    except Exception:
        auc = 0.0
    return {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1, 'roc_auc': auc}

args = TrainingArguments(
    output_dir=str(OUT_DIR),
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    report_to='none',
    # ── Kaggle GPU optimisations ──
    fp16=True,                                   # mixed-precision (Kaggle GPUs support FP16)
    dataloader_num_workers=2,                    # parallel data loading
    gradient_accumulation_steps=2,               # effective batch = 16
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()
metrics = trainer.evaluate(test_ds)
metrics

In [ ]:
ADAPTER_DIR  = OUT_DIR / 'adapter'
TOKENIZER_DIR = OUT_DIR / 'tokenizer'

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(TOKENIZER_DIR)
print('Saved adapter  →', ADAPTER_DIR)
print('Saved tokenizer →', TOKENIZER_DIR)

## Downloading results from Kaggle

After the notebook run completes, the trained LoRA adapter and tokenizer are saved under
`/kaggle/working/codebert_lora/`. You can:

1. **Download directly** — Click *Output* tab → download `codebert_lora/` folder.
2. **Use as Notebook Output** — Add this notebook's output as input to a downstream notebook.
3. **Push to Kaggle Datasets** — Use the Kaggle API: `kaggle datasets create -p /kaggle/working/codebert_lora`.